# 1、PIIMiddleware中间件

## 举例1：使用内置检测器

In [1]:
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [2]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)
response = agent.invoke({

    "messages" : [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

抱歉，我不能替你执行这些涉及外部操作或敏感信息的请求，包括：

- 向某个邮箱发送邮件
- 查询银行卡余额
- 访问你提供的链接
- 确认或处理看起来像 MAC 地址/设备标识这类敏感信息

如果你愿意，我可以帮你：

1. **起草邮件内容**，你复制到邮箱里发送；
2. **说明如何安全查询银行卡余额**（通过官方 App / 网银 / 客服）；
3. **帮你判断一段字符串是否像 MAC 地址**，但不会对敏感标识做实际查询；
4. **分析链接的安全性**，如果你把网页内容贴出来，我可以帮你看看是否可疑。

例如，这个格式 `**-**-**-**-**-11` **看起来不完整，不能直接确认是标准 MAC 地址**。标准 MAC 地址通常是 6 组十六进制字符，例如：

- `AA:BB:CC:DD:EE:FF`
- `AA-BB-CC-DD-EE-FF`

如果你想，我可以现在直接帮你**写一封邮件草稿**。


In [3]:
try:
    response1 = agent.invoke({
            "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到ip，抛出异常{e}")

检测到ip，抛出异常Detected 1 instance(s) of ip in text content


## 举例2：自定义检测器/函数

In [4]:
import re

# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

In [5]:
text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
    
================================== Ai Message ==================================

我不能帮你验证、使用或泄露 API 密钥，也不能替你拨打电话。

另外，`https://localhost:12345` 是你本机的回环地址，只能在你自己的设备上访问；如果你想我帮你检查它是否可用，我可以告诉你如何在本机安全测试。

如果你愿意，我可以继续帮你：
- 判断一个密钥格式是否“看起来像” API key（不涉及实际验证）
- 教你如何安全地在本机检查 `localhost:12345`
- 帮你写一段拨号或 API 调用的示例代码（不包含敏感信息）
